In [1]:
import numpy as np

r_left = np.array([1.0,0.0])
print(r_left)

r_right = np.array([0.0,1.0])
print(r_right)

x = np.array([-1.0, 0.0])
print(x)

score_left = np.dot(r_left,x)
print(score_left)

score_right = np.dot(r_right,x)
print(score_right)

if score_left > score_right:
    task_id = "LEFT_SAMPLE"
else:
    task_id = "RIGHT_SAMPLE"

print(task_id)

margin = abs(score_left - score_right)
print(f"{margin:.4f}")

best_score = max(score_left, score_right)

if best_score >= 0.6 and margin >= 0.04:
  decision = task_id
else:
  decision = "REJECT"

print("最終判定:",decision)



[1. 0.]
[0. 1.]
[-1.  0.]
-1.0
0.0
RIGHT_SAMPLE
1.0000
最終判定: REJECT


## 第1回演習：コサイン類似度とREJECT判定
## 1. 実験目的

2次元ベクトルを用いてコサイン類似度を計算し、作業IDの選択方法を理解する。また、最大類似度と候補間差の2つの閾値によるREJECT判定の仕組みを検証する。

### 2. 実験条件

基準ベクトル：
- LEFT_SAMPLE = (1.0, 0.0)
- RIGHT_SAMPLE = (0.0, 1.0)

判定閾値：
- 最小類似度：0.60
- 最小候補間差：0.04

### 3. 実験結果

| 入力ベクトル | 左類似度 | 右類似度 | margin | 最終判定 |
|---|---:|---:|---:|---|
| (0.8, 0.6) | 0.8000 | 0.6000 | 0.2000 | LEFT_SAMPLE |
| (0.7071, 0.7071) | 0.7071 | 0.7071 | 0.0000 | REJECT |
| (-1.0, 0.0) | -1.0000 | 0.0000 | 1.0000 | REJECT |

### 4. 考察
実験①では、最大類似度0.8000、候補間差0.2000となり、いずれも設定した閾値を満たしたため、LEFT_SAMPLEが選択された。

実験②と実験③はいずれもREJECTとなったが、その原因は異なる。

実験②では、左右の類似度がともに約0.7071で、最大類似度は判定閾値0.60を上回っていた。しかし、両者の類似度が等しく、候補間差が0.0000となったため、最小候補間差0.04の条件を満たさず、REJECTとなった。

一方、実験③では、候補間差が1.0000と大きく、最小候補間差0.04の条件を満たしていた。しかし、最大類似度が0.0000であり、判定閾値0.60を下回ったため、REJECTとなった。

以上の結果から、最大類似度の閾値は登録された作業候補との類似性を確認するために、候補間差の閾値は複数の作業候補を区別できるかを確認するために、それぞれ異なる役割を持つことが分かった。

したがって、類似度が最も高い作業IDを選択するだけでなく、2つの条件を組み合わせて採用・拒否を判断することが重要である。

ただし、今回の実験は人工的に設定した2次元ベクトルを用いた基礎検証であり、実際の日本語指示に対するAIの認識精度やロボット動作の安全性を保証するものではない。

In [7]:
test_vectors = [
    np.array([0.8, 0.6]),
    np.array([1.0, 1.0]) / np.sqrt(2),
    np.array([-1.0, 0.0])
]

print(len(test_vectors))

# 各入力に対する期待判定
expected_results = ["LEFT_SAMPLE", "REJECT", "REJECT"]

for i, x in enumerate(test_vectors):
    score_left = np.dot(r_left, x)
    score_right = np.dot(r_right, x)

    best_score = max(score_left, score_right)
    margin = abs(score_left - score_right)

    if score_left > score_right:
        task_id = "LEFT_SAMPLE"
    else:
        task_id = "RIGHT_SAMPLE"

    if best_score >= 0.60 and margin >= 0.04:
        decision = task_id
    else:
        decision = "REJECT"

    print("入力:", x)
    print(f"最大類似度={best_score:.4f}, margin={margin:.4f}")
    print("最終判定:", decision)
    assert decision == expected_results[i]

3
入力: [0.8 0.6]
最大類似度=0.8000, margin=0.2000
最終判定: LEFT_SAMPLE
入力: [0.70710678 0.70710678]
最大類似度=0.7071, margin=0.0000
最終判定: REJECT
入力: [-1.  0.]
最大類似度=0.0000, margin=1.0000
最終判定: REJECT
